# Chapter 1 — What Does It Mean to Debug?

**Book alignment:** Debugging AI From First Principles, Chapter 1

**Question this notebook isolates:** Two different one-line edits both make the failing
€1,200 invoice pass — does an ordered checkpoint table plus a counterfactual intervention
tell you which one is on the causal path, where *"it works now"* cannot?

Constructed illustration, deterministic, standard library only. No LM is called. The book
is the authority for chapter meaning; the cells below make its mechanism runnable and
falsifiable.

In [ ]:
import random

# The billing function, with the defect the chapter describes: a stray `discount = 0.0`
# left over from a merge, sitting two lines below the (correct) discount branch.
# `discount_override` and `render_fudge` are counterfactual hooks, not production knobs.
def invoice_total(items, *, threshold=1000, rate=0.10,
                  discount_override=None, render_fudge=1.0):
    subtotal = sum(i["price"] * i["qty"] for i in items)
    discount = subtotal * rate if subtotal > threshold else 0.0
    discount = 0.0                                   # <-- the defect
    if discount_override is not None:
        discount = discount_override
    total = subtotal - discount
    tax = total * 0.20
    return round((total + tax) * render_fudge, 2)


# Intent, written down first — a pure re-statement of the spec, no defect.
def intended(items, *, threshold=1000, rate=0.10):
    subtotal = sum(i["price"] * i["qty"] for i in items)
    discount = subtotal * rate if subtotal > threshold else 0.0
    total = subtotal - discount
    return {"subtotal": round(subtotal, 2), "discount": round(discount, 2),
            "total": round(total, 2), "invoiced": round(total * 1.20, 2)}

## 1. Reproduce the failure — deterministically

REPRODUCE before anything else: a failure you cannot retrigger on demand caps every later
conclusion at speculation.

In [ ]:
order = [{"price": 600, "qty": 1}, {"price": 600, "qty": 1}]   # a €1,200 invoice
observed = invoice_total(order)
expected = intended(order)["invoiced"]
print(f"observed = {observed}   expected = {expected}")

assert (observed, expected) == (1440.0, 1296.0)
assert all(invoice_total(order) == 1440.0 for _ in range(5))   # 5/5 — stable, not flaky
print("reproduced 5/5: the wrong number is the same every run")

## 2. "It works now" cannot separate three hypotheses

Three mechanisms all produce *wrong invoice amount*:

- **H1** the discount condition is wrong (boundary / comparison error)
- **H2** the subtotal computation is wrong
- **H3** the discount is computed, then dropped downstream

Only evidence at ordered intermediate points separates them — each predicts a *different
first* wrong value.

In [ ]:
def checkpoints(items):
    subtotal = sum(i["price"] * i["qty"] for i in items)
    cond = subtotal > 1000
    discount_in_branch = subtotal * 0.10 if cond else 0.0
    discount_after = 0.0                                   # what the real function leaves
    return [
        ("subtotal",                round(subtotal, 2),          1200.0),
        ("condition subtotal>1000", cond,                        True),
        ("discount inside branch",  round(discount_in_branch, 2), 120.0),
        ("discount after branch",   round(discount_after, 2),     120.0),
    ]

rows = checkpoints(order)
first_div = next(name for name, got, want in rows if got != want)
for name, got, want in rows:
    mark = "   <-- FIRST DIVERGENCE" if name == first_div else ""
    print(f"{name:24}  got={got!r:<7}  intended={want!r}{mark}")

assert first_div == "discount after branch"
# H1 would first diverge at 'discount inside branch' (it matches -> H1 out)
# H2 would first diverge at 'subtotal'               (it matches -> H2 out)
# H3: right inside the branch, wrong after           -> the surviving hypothesis
assert rows[2][1] == rows[2][2] and rows[3][1] != rows[3][2]
print(f"\nfirst divergence names H3; the final €{observed} names none of them")

## 3. The counterfactual: force the suspect value, watch the failure move

A difference is a *lead*. It earns the word **cause** only when an intervention that
changes it — and nothing else — changes the failure as predicted.

In [ ]:
obs = invoice_total(order)
cf  = invoice_total(order, discount_override=120.0)     # set discount to its intended value
print(f"observed         -> {obs}")
print(f"discount := 120.0 -> {cf}")

assert (obs, cf) == (1440.0, 1296.0)
# forcing an *unrelated* knob does not move the failure:
assert invoice_total(order, render_fudge=1.0) == 1440.0
print("discount is on the causal path (forward intervention); render rounding is not")

## 4. A downstream "fix" hides the symptom without touching the cause

Developer A believes it is float noise and patches the render arithmetic so *this* invoice
matches. The ticket looks closed — and the report is "wrong again by a different amount"
the next night.

In [ ]:
FUDGE = 1296.0 / 1440.0
assert invoice_total(order, render_fudge=FUDGE) == 1296.0        # this invoice now "passes"

# but the first divergence is still at 'discount after branch' ...
assert checkpoints(order)[3][1] != checkpoints(order)[3][2]

# ... and an invoice that never triggered the discount bug is now wrong in a new way:
order2 = [{"price": 300, "qty": 1}, {"price": 300, "qty": 1}]    # €600 - below the threshold
got2, want2 = invoice_total(order2, render_fudge=FUDGE), intended(order2)["invoiced"]
print(f"below-threshold invoice: got={got2}  expected={want2}  match={got2 == want2}")

assert invoice_total(order2) == want2                           # the *unpatched* code was right here
assert got2 != want2                                            # the render patch broke it
print("symptom relief on one input; the cause transition (discount dropped) is untouched,")
print("and the downstream patch is now a second defect on the no-discount path")

## 5. Minimization is a sequence of experiments, not tidying

`minimize_failure` shrinks a failing input by *repeatedly testing smaller variants*. Each
`still_fails` verdict is one discriminating experiment; the small list is the by-product.
(An educational simplification of Zeller & Hildebrandt's `ddmin`.)

In [ ]:
def minimize_failure(parts, still_fails):
    current, granularity = list(parts), 2
    while len(current) >= 2:
        chunk = max(1, len(current) // granularity)
        for start in range(0, len(current), chunk):
            candidate = current[:start] + current[start + chunk:]
            if candidate and still_fails(candidate):
                current, granularity = candidate, max(2, granularity - 1)
                break
        else:
            if granularity >= len(current):
                break
            granularity = min(len(current), granularity * 2)
    return current


trials = {"n": 0}
def still_fails(items):
    trials["n"] += 1
    return round(invoice_total(items), 2) != intended(items)["invoiced"]

padded = [{"price": 40, "qty": 1}] * 18 + [{"price": 600, "qty": 1}] * 2
minimal = minimize_failure(padded, still_fails)
print(f"start {len(padded)} items  ->  minimal {len(minimal)} items"
      f"   ({trials['n']} candidate lists tested)")

assert still_fails(minimal) and len(minimal) <= 4
print("each still_fails() call was an experiment removing a set of circumstances from suspicion")

## 6. A flaky oracle sends minimization to the wrong element

If `still_fails` is noisy, a single trial per verdict is not enough — and minimization
happily isolates an element that is not the culprit. House floor: repeat each verdict until
it is stable (≥ 3 trials).

In [ ]:
_rng = random.Random(0)
def flaky_still_fails(items):
    real = still_fails(items)
    return False if (real and _rng.random() < 0.35) else real     # 35% false "passes"

trials["n"] = 0
bad = minimize_failure(padded, flaky_still_fails)
prices = sorted(i["price"] for i in bad)
print(f"flaky oracle -> minimal prices {prices}   (a faithful run isolates [600, 600])")
if prices != [600, 600]:
    print("the noisy verdict dropped a load-bearing item: minimization convicted the wrong element")
else:
    print("this seed happened to survive; the hazard is real regardless — repeat noisy verdicts")

## What we earned

Two edits closed the €1,200 ticket. The ordered checkpoint table and one counterfactual
intervention show that only the stray reassignment is on the causal path; a second invoice
exposes the render patch as symptom relief. Minimization is a chain of pass/fail
experiments — and a noisy oracle corrupts it.

**Notebook 02 / Chapter 2** turns *"find the first divergence"* into a bisection procedure
over a five-stage pipeline — and shows the one precondition (monotonicity) that bisection
silently needs.